# GigSmart AI — ML Model Training

## Objective
Train and compare multiple ML models to:
1. Predict best app (Classification)
2. Predict expected net profit (Regression)

## Models
- Classification: Logistic Regression, 
  Decision Tree, Random Forest
- Regression: Linear Regression,
  Decision Tree, Random Forest# GigSmart AI — ML Model Training

## Objective

Train and compare multiple machine learning models to solve both business prediction tasks.

### Prediction Goals

1. Predict best app for maximum profit *(Classification)*  
2. Predict expected net profit *(Regression)*

---

## Models Used

### Classification Models
- Logistic Regression  
- Decision Tree Classifier  
- Random Forest Classifier  

### Regression Models
- Linear Regression  
- Decision Tree Regressor  
- Random Forest Regressor  

---

## Workflow

- Load processed dataset  
- Feature selection  
- Encode categorical data  
- Train-test split  
- Train models  
- Evaluate performance  
- Compare results  
- Select best model


In [1]:
# Libraries import karo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, f1_score, 
                             precision_score, recall_score,
                             classification_report,
                             mean_absolute_error, 
                             mean_squared_error, r2_score)

print("All libraries imported successfully ")

All libraries imported successfully 


In [3]:
# Dataset load karo
df = pd.read_csv('../data/processed/gigsmart_dataset.csv')

print("Dataset loaded ✅")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

Dataset loaded ✅
Shape: (45593, 16)

Columns: ['hour', 'day_of_week', 'is_weekend', 'is_festival', 'weather', 'zone_type', 'is_lunch_time', 'is_dinner_time', 'distance_km', 'estimated_time_min', 'app_name', 'expected_payout', 'fuel_cost', 'time_cost', 'net_profit', 'best_app']

First 3 rows:


,hour,day_of_week,is_weekend,is_festival,weather,zone_type,is_lunch_time,is_dinner_time,distance_km,estimated_time_min,app_name,expected_payout,fuel_cost,time_cost,net_profit,best_app
0,11,5,1,0,clear,normal,0,0,4.2,24,Swiggy,101.55,21.0,36.0,44.55,Swiggy
1,19,4,0,0,rain,low,0,1,14.5,33,Blinkit,220.01,72.5,49.5,98.01,Swiggy
2,8,5,1,0,fog,normal,0,0,6.4,26,Blinkit,104.92,32.0,39.0,33.92,Swiggy


## Categorical Encoding

ML models cannot understand text values.
We convert categorical columns to numbers
using Label Encoding.

- **weather**: clear=0, fog=1, rain=2
- **zone_type**: busy=0, low=1, normal=2
- **app_name**: Blinkit=0, Swiggy=1, Zomato=2
- **best_app**: Blinkit=0, Swiggy=1, Zomato=2

In [4]:
# Categorical columns encode karo
le = LabelEncoder()

# Weather encode karo
df['weather'] = le.fit_transform(df['weather'])
print("Weather encoding:")
print(dict(zip(le.classes_, 
               le.transform(le.classes_))))

# Zone type encode karo
df['zone_type'] = le.fit_transform(df['zone_type'])
print("\nZone type encoding:")
print(dict(zip(le.classes_,
               le.transform(le.classes_))))

# App name encode karo
df['app_name'] = le.fit_transform(df['app_name'])
print("\nApp name encoding:")
print(dict(zip(le.classes_,
               le.transform(le.classes_))))

# Best app encode karo (target variable)
le_target = LabelEncoder()
df['best_app'] = le_target.fit_transform(df['best_app'])
print("\nBest app encoding (TARGET):")
print(dict(zip(le_target.classes_,
               le_target.transform(le_target.classes_))))

print("\nEncoding done ✅")
print(df.head(3))

Weather encoding:
{'clear': np.int64(0), 'fog': np.int64(1), 'rain': np.int64(2)}

Zone type encoding:
{'busy': np.int64(0), 'low': np.int64(1), 'normal': np.int64(2)}

App name encoding:
{'Blinkit': np.int64(0), 'Swiggy': np.int64(1), 'Zomato': np.int64(2)}

Best app encoding (TARGET):
{'Blinkit': np.int64(0), 'Swiggy': np.int64(1), 'Zomato': np.int64(2)}

Encoding done ✅
   hour  day_of_week  is_weekend  is_festival  weather  zone_type  \
0    11            5           1            0        0          2   
1    19            4           0            0        2          1   
2     8            5           1            0        1          2   

   is_lunch_time  is_dinner_time  distance_km  estimated_time_min  app_name  \
0              0               0          4.2                  24         1   
1              0               1         14.5                  33         0   
2              0               0          6.4                  26         0   

   expected_payout  fuel_cost 

### Observations
- All categorical columns successfully converted 
  to numerical format
- weather: clear=0, fog=1, rain=2
- zone_type: busy=0, low=1, normal=2
- app_name and best_app: Blinkit=0, Swiggy=1, Zomato=2

### Insight
Label Encoding assigns numbers based on 
alphabetical order. Tree-based models like 
Decision Tree and Random Forest work well 
with label encoded data as they split on 
values, not distances.

## Feature Selection

Selecting input features (X) and target variables (y) for both machine learning tasks.

#Target variables

- **Classification Target:** `best_app`  
- **Regression Target:** `net_profit`

#Input Features

All remaining useful columns are used as input features after removing target columns.

#Purpose

- `X` contains independent variables used for prediction  
- `y_classification` used to predict best app  
- `y_regression` used to predict expected profit

In [5]:
# Input features
X = df.drop(columns=['best_app', 'net_profit'])

# Classification target
y_class = df['best_app']

# Regression target
y_reg = df['net_profit']

print("Input Features (X):")
print(X.columns.tolist())

print("\nX shape:")
print(X.shape)

print("\nClassification target (y_class):")
print(y_class.value_counts())

print("\nRegression target (y_reg):")
print(y_reg.describe().round(2))

Input Features (X):
['hour', 'day_of_week', 'is_weekend', 'is_festival', 'weather', 'zone_type', 'is_lunch_time', 'is_dinner_time', 'distance_km', 'estimated_time_min', 'app_name', 'expected_payout', 'fuel_cost', 'time_cost']

X shape:
(45593, 14)

Classification target (y_class):
best_app
0    16007
2    15794
1    13792
Name: count, dtype: int64

Regression target (y_reg):
count    45593.00
mean        43.54
std         26.40
min        -28.90
25%         24.51
50%         41.48
75%         60.99
max        134.97
Name: net_profit, dtype: float64


## Observation

- Total input records available: **45,593**
- Total input features used for training: **14**
- Dataset is sufficient for reliable model training
- Classification target contains **3 classes** (Best App options)
- Class distribution is fairly balanced:
  - Blinkit: 16,007
  - Zomato: 15,794
  - Swiggy: 13,792
- Regression target (`net_profit`) has both positive and negative values

## Insights

- Balanced class distribution helps classification models learn fairly without strong bias.
- Blinkit appears slightly dominant in profitability scenarios.
- Average net profit is **₹43.54**, indicating most orders are profitable.
- Median profit (**₹41.48**) is close to mean, showing stable profit distribution.
- Minimum profit is **-₹28.90**, meaning some orders lead to losses.
- High maximum profit (**₹134.97**) shows strong earning opportunities in favorable conditions.
- Dataset is suitable for both classification and regression modeling.

## Train Test Split

We split the dataset into training and testing sets for both tasks:

- **Classification:** Predict best_app  
- **Regression:** Predict net_profit  

### Purpose

- Training set → used to train models  
- Testing set → used to evaluate model performance  
- Test size = 20%  
- Random state = 42 for reproducibility

In [6]:
from sklearn.model_selection import train_test_split

# Input Features
X = df.drop(columns=['best_app', 'net_profit'])

# Classification Target
y_class = df['best_app']

# Regression Target
y_reg = df['net_profit']


# ===============================
# Classification Split
# ===============================
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X,
    y_class,
    test_size=0.20,
    random_state=42,
    stratify=y_class
)


# ===============================
# Regression Split
# ===============================
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X,
    y_reg,
    test_size=0.20,
    random_state=42
)


print("Classification Split:")
print("X_train:", X_train_class.shape)
print("X_test :", X_test_class.shape)
print("y_train:", y_train_class.shape)
print("y_test :", y_test_class.shape)

print("\nRegression Split:")
print("X_train:", X_train_reg.shape)
print("X_test :", X_test_reg.shape)
print("y_train:", y_train_reg.shape)
print("y_test :", y_test_reg.shape)

Classification Split:
X_train: (36474, 14)
X_test : (9119, 14)
y_train: (36474,)
y_test : (9119,)

Regression Split:
X_train: (36474, 14)
X_test : (9119, 14)
y_train: (36474,)
y_test : (9119,)


## Observations

- Dataset successfully split into training and testing sets.
- 80% data used for training and 20% used for testing.
- Total 14 input features used for model learning.
- Separate splits created for classification and regression tasks.

## Insight

Proper train-test split helps evaluate model performance on unseen data and reduces overfitting risk.